# YuvaIntern Task 6 — Final Data Science Capstone Project
## Bank Marketing Campaign Analysis and Subscription Prediction

**Business problem:** Help a bank understand which customer/campaign patterns are associated with term-deposit subscription and build a baseline model to support campaign prioritization.

**Dataset:** UCI Bank Marketing (dataset ID 222)  
**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn, scikit-learn, ucimlrepo

**Scope and caution:** This is a public historical dataset. Model output is for learning and decision support, not an automatic decision about a real customer. The dataset includes a field `duration` that is only known after a call ends; it is excluded from the primary predictive model to avoid using information unavailable before/during campaign targeting. The notebook compares metrics and explains limitations.

## 1. Problem statement, objectives, and success criteria

### Problem statement
Bank marketing campaigns use resources to contact potential customers. The goal is to analyze historical campaign data, understand patterns associated with a successful term-deposit subscription, and evaluate a baseline model that may help prioritize future outreach.

### Objectives
1. Load and document a reliable public dataset.
2. Assess data types, missing values, duplicates, distributions, and class imbalance.
3. Prepare features and define the target variable.
4. Perform exploratory data analysis and create meaningful charts.
5. Train and evaluate a baseline classification model.
6. Interpret results, limitations, and practical recommendations.

### Success criteria
- Reproducible notebook that runs top to bottom.
- Data-quality checks and explicit preprocessing decisions.
- Evaluation using precision, recall, F1-score, confusion matrix, and ROC-AUC where supported.
- Recommendations tied to the analysis, without claiming causation.

In [ ]:
# Install if needed:
# %pip install ucimlrepo pandas numpy matplotlib seaborn scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
    RocCurveDisplay
)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")


## 2. Load the official dataset

In [ ]:
bank = fetch_ucirepo(id=222)
X_raw = bank.data.features.copy()
y_raw = bank.data.targets.copy()

# Combine features and target into one analysis frame.
target_col = y_raw.columns[0]
df = X_raw.copy()
df[target_col] = y_raw[target_col].values

print("Dataset shape:", df.shape)
print("Target column:", target_col)
display(df.head())
display(df.sample(min(5, len(df)), random_state=42))


## 3. Data understanding and quality assessment

In [ ]:
print("Columns:", df.columns.tolist())
display(df.dtypes.to_frame("dtype"))
display(df.describe(include="all").T)

quality = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique(dropna=True)
}).sort_values("missing_count", ascending=False)
display(quality)
print("Exact duplicate rows:", int(df.duplicated().sum()))
print("\nTarget distribution:")
display(df[target_col].value_counts(dropna=False).to_frame("count"))
display((df[target_col].value_counts(normalize=True, dropna=False) * 100).round(2).to_frame("percent"))


## 4. Data preparation and feature decisions

The UCI Bank Marketing dataset has versions. This notebook uses the repository's dataset ID 222, which generally provides the bank marketing features and a binary subscription target. The code below normalizes the target to 0/1, removes exact duplicates for the modeling copy, and excludes `duration` from the primary model because call duration is not available before a call is made. Other feature names are taken from the loaded data rather than hard-coded beyond this documented decision.

In [ ]:
data = df.copy()

# Normalize target to binary values for common UCI representations (yes/no).
target_values = data[target_col].astype(str).str.strip().str.lower()
unique_targets = set(target_values.dropna().unique())
if unique_targets <= {"yes", "no"}:
    data["target_subscription"] = target_values.map({"yes": 1, "no": 0})
else:
    # Handle numeric or other binary target representations safely.
    target_numeric = pd.to_numeric(data[target_col], errors="coerce")
    if target_numeric.notna().all() and target_numeric.nunique() == 2:
        vals = sorted(target_numeric.unique())
        data["target_subscription"] = target_numeric.map({vals[0]: 0, vals[1]: 1})
    else:
        raise ValueError(f"Unexpected target values: {sorted(unique_targets)}")

# Remove exact duplicate rows from modeling copy.
before = len(data)
data = data.drop_duplicates().copy()
print("Rows before duplicate removal:", before)
print("Rows after duplicate removal:", len(data))

# Find duration field irrespective of capitalization, if present.
duration_cols = [c for c in data.columns if c.lower() == "duration"]
excluded_cols = [target_col, "target_subscription"] + duration_cols

# Drop original target and leakage-prone duration from predictors.
X = data.drop(columns=[c for c in excluded_cols if c in data.columns])
y = data["target_subscription"]

# Identify numeric and categorical predictors.
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)
print("Duration excluded:", duration_cols)
print("Positive target rate (%):", round(y.mean() * 100, 2))


## 5. Exploratory data analysis

In [ ]:
# Target balance
target_counts = y.value_counts().rename(index={0: "No subscription", 1: "Subscription"})
display(target_counts.to_frame("records"))

plt.figure(figsize=(7, 4))
plt.bar(target_counts.index.astype(str), target_counts.values)
plt.title("Subscription Target Distribution")
plt.xlabel("Target class")
plt.ylabel("Records")
plt.tight_layout()
plt.show()


In [ ]:
# Campaign/contact numeric feature summaries, where available
candidate_num = [c for c in numeric_features if c.lower() in {"age", "campaign", "previous", "pdays"}]
if candidate_num:
    display(data.groupby("target_subscription")[candidate_num].mean().T.rename(columns={0:"No subscription",1:"Subscription"}))
else:
    print("No expected numeric campaign fields found; available numeric fields:", numeric_features)


In [ ]:
# Target rate by selected categorical features, where available
for candidate in ["job", "marital", "education", "contact", "month", " poutcome", "poutcome"]:
    if candidate in data.columns:
        rate = data.groupby(candidate, dropna=False)["target_subscription"].agg(["mean", "count"]).sort_values("mean", ascending=False)
        rate["subscription_rate_percent"] = rate["mean"] * 100
        display(rate.drop(columns="mean").head(15))


In [ ]:
# Age distribution, if age is available
age_cols = [c for c in numeric_features if c.lower() == "age"]
if age_cols:
    age_col = age_cols[0]
    plt.figure(figsize=(8, 4))
    sns.histplot(data=data, x=age_col, hue="target_subscription", bins=30, element="step", stat="density", common_norm=False)
    plt.title("Age Distribution by Subscription Outcome")
    plt.tight_layout()
    plt.show()


## 6. Train/test split and preprocessing pipeline

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=2000, class_weight="balanced"))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]


## 7. Model evaluation

In [ ]:
metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "F1-score": f1_score(y_test, y_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_prob)
}
display(pd.Series(metrics, name="score").to_frame())
print(classification_report(y_test, y_pred, zero_division=0))


In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Predicted no", "Predicted yes"],
            yticklabels=["Actual no", "Actual yes"])
plt.title("Confusion Matrix — Logistic Regression")
plt.xlabel("Prediction")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

RocCurveDisplay.from_predictions(y_test, y_prob)
plt.title("ROC Curve — Logistic Regression")
plt.tight_layout()
plt.show()


## 8. Interpretable findings generated from the data

In [ ]:
# Calculate positive rates for available categorical fields.
for candidate in ["job", "marital", "education", "contact", "month", "poutcome"]:
    if candidate in data.columns:
        summary = data.groupby(candidate, dropna=False)["target_subscription"].agg(
            records="size", subscription_rate="mean"
        )
        summary["subscription_rate_percent"] = summary["subscription_rate"] * 100
        print(f"Subscription rate by {candidate}:")
        display(summary.sort_values("subscription_rate", ascending=False).drop(columns="subscription_rate").head(10))

print("Model metrics:")
for key, value in metrics.items():
    print(f"{key}: {value:.3f}")


## 9. Business recommendations

Use the observed group-level subscription rates and model evaluation to guide further investigation, not to make automatic customer decisions.

1. Compare campaign outcomes across contact channels, months, job groups, and previous campaign outcomes where the data supports these fields.
2. Prioritize follow-up experiments for groups with higher observed subscription rates, while validating that group sizes are adequate.
3. Review precision and recall together. A campaign may value recall when missing likely subscribers is costly, but precision matters when contact resources are limited.
4. Do not use call duration in pre-call targeting because it is unavailable until after the interaction.
5. Evaluate campaign strategies with controlled experiments and monitor performance over time.
6. Review fairness, privacy, consent, and applicable marketing rules before deploying any model.

## 10. Limitations and future scope

- Historical campaign data may not reflect current customer behavior or current campaign design.
- The dataset may have class imbalance; accuracy alone is not enough.
- Logistic Regression is a baseline, not necessarily the best-performing model.
- Excluding duration makes the model more suitable for pre-call prioritization but may reduce predictive performance.
- Association does not prove causation.
- Future work: compare Random Forest/Gradient Boosting, tune thresholds using business costs, cross-validate, calibrate probabilities, and assess subgroup performance.

## 11. Conclusion

This capstone defines a business problem, loads and prepares public data, performs EDA, builds a leakage-aware preprocessing and modeling pipeline, evaluates classification performance, and translates findings into practical recommendations. All calculated metrics and charts are generated when the notebook runs.